<a href="https://colab.research.google.com/github/dnhshl/cc-ai/blob/main/cv_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision (YOLO & SAM)
### KI in der Robotik

Moderne **Deep Learning Modelle**, können in Echtzeit Objekte erkennen und segmentieren.

Wir vergleichen zwei Ansätze:
1.  **YOLO (You Only Look Once):** Liefert Bounding Boxes (Rechtecke) und Segmentierungen für die Hindernisvermeidung.
2.  **SAM (Segment Anything Model):** Liefert pixelgenaue Masken für präzises Greifen (Grasping).

26.01.2026 DN

In [ ]:
# 1. Installation & Setup
!pip install ultralytics

from ultralytics import YOLO
from ultralytics import FastSAM
import requests
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np



In [ ]:
# Lade ein vortrainiertes YOLO-Modell (Nano-Größe für Speed)
model_yolo= YOLO('yolo11m.pt')
#model_yolo_medium = YOLO('yolo11m.pt')

# Lade das Yolo-Modell für Segmentation
model_seg = YOLO('yolo11m-seg.pt') # 'seg' steht für Segmentation


In [ ]:
import requests
# --- BILDER-BIBLIOTHEK ---

urls = {
    "kueche": "https://images.pexels.com/photos/534151/pexels-photo-534151.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1",
    "verkehr": "https://images.pexels.com/photos/8816147/pexels-photo-8816147.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1",
    "schreibtisch": "https://images.pexels.com/photos/32092254/pexels-photo-32092254.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1",
}

# 2. Speicher für die Bilder
bild_speicher = {}

print("--- Lade Bilder in den RAM ---")
for name, url in urls.items():
    try:
        # Request mit Browser-Header (gegen 403 Fehler)
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()

        # Umwandlung: Bytes -> NumPy Array -> Bild-Matrix
        image_array = np.asarray(bytearray(resp.content), dtype=np.uint8)
        img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

        if img is not None:
            bild_speicher[name] = img
            print(f"✅ {name}: Erfolgreich im Speicher.")
        else:
            print(f"❌ {name}: Konnte nicht dekodiert werden.")

    except Exception as e:
        print(f"⚠️ Fehler bei {name}: {e}")


In [ ]:
img_data = bild_speicher["verkehr"]


## 2. YOLO: Bounding Box Detection
YOLO ist der Standard für Echtzeit-Objekterkennung in der Robotik. Es teilt das Bild in ein Gitter und sagt für jede Zelle vorher, ob und wo sich ein Objekt befindet. Dies geschieht in einem einzigen Durchlauf des neuronalen Netzes.

**Anwendung:** Hindernisvermeidung, Objektlokalisierung.

In [ ]:

# Führe die Objekterkennung durch
results_yolo = model_yolo.predict(img_data, conf=0.5) # Konfidenz-Schwelle 50%

# Visualisiere das Ergebnis
for r in results_yolo:
    # Zeichne Bounding Boxes und Labels ins Bild
    res_plotted = r.plot()
    cv2_imshow(res_plotted)

## 3. Yolo Segmentation
Für das **Greifen (Grasping)** ist eine Bounding Box oft nicht genau genug. Der Roboter muss wissen, wo die exakten Kanten des Objekts sind, um seine Finger korrekt zu platzieren.


**Anwendung:** Präzises Greifen, Qualitätskontrolle.

In [ ]:

# Führe die Segmentation durch
results_seg = model_seg.predict(img_data, conf=0.5)

# Visualisiere das Ergebnis
for r in results_seg:
    # Zeichne die farbigen Segmentierungsmasken
    seg_plotted = r.plot()
    cv2_imshow(seg_plotted)